In [250]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [140]:
# Load the data
df = pd.read_csv('data/apple_sales_data_transformed.csv')

In [141]:
df.head()

,sale_id,sale_date,year,quarter,month,country,region,city,product_name,category,...,revenue_local_currency,sales_channel,payment_method,customer_segment,customer_age_group,previous_device_os,customer_rating,return_status,month_calender,revenue_no_discount_usd
0,APPL-00000001,2022-01-03,2022,Q1,January,Argentina,South America,Buenos Aires,AirPods (3rd Gen),AirPods,...,134344.84,Third-Party Retailer,Cash,Government,45–54,NaN,4.1,Kept,2022-01-01,159.27
1,APPL-00000002,2022-01-04,2022,Q1,January,Argentina,South America,Buenos Aires,USB-C Woven Charge Cable,Accessories,...,115597.15,Authorized Reseller,Debit Card,Business,45–54,NaN,4.8,Kept,2022-01-01,149.95
2,APPL-00000003,2022-05-18,2022,Q2,May,Argentina,South America,Buenos Aires,Apple Watch Series 8,Apple Watch,...,1066341.76,Corporate / B2B,Credit Card,Individual,18–24,NaN,4.3,Kept,2022-05-01,1175.68
3,APPL-00000004,2022-05-23,2022,Q2,May,Argentina,South America,Buenos Aires,MacBook Pro 14-inch (M3),Mac,...,3506044.78,Carrier Store,Credit Card,Education,45–54,NaN,NaN,Kept,2022-05-01,3865.54
4,APPL-00000005,2022-07-13,2022,Q3,July,Argentina,South America,Buenos Aires,Apple Watch Ultra 2,Apple Watch,...,1952780.07,Apple Store,Net Banking,Education,18–24,NaN,NaN,Kept,2022-07-01,2266.32


# Executive overview

In [216]:
total_revenue = df['revenue_usd'].sum()
total_units = df['units_sold'].sum()
avg_discount = df['discount_pct'].mean()

kpis = pd.DataFrame({
    "Metric": ["Total Revenue", "Units Sold", "Average Discount"],
    "Value": [
        f"${total_revenue/1e6:.1f}M",
        f"{total_units:,}",
        f"{avg_discount:.1f}%"
    ]
})

ind1 = go.Figure(go.Indicator(
    mode="number",
    value=total_revenue,
    number={'prefix': "$", 'valueformat': ",.0f"},
    title={"text": "Total Revenue between 2022 and 2025"}
))

ind2 = go.Figure(go.Indicator(
    mode="number",
    value=total_units,
    number={'valueformat': ".0f"},
    title={"text": "Total Units Sold between 2022 and 2025"}
))

ind3 = go.Figure(go.Indicator(
    mode="number",
    value=avg_discount,
    number={'suffix': "%", 'valueformat': ".1f"},
    title={"text": "Average Discount Percentage between 2022 and 2025"}
))

ind1.show()
ind2.show()
ind3.show()

In [ ]:
# Line chart of revenue over time

df_fig1 = df.groupby('month_calender')['revenue_usd'].sum().reset_index()
fig1 = px.line(df_fig1, x = 'month_calender', 
               y = 'revenue_usd', 
               markers=True,
               labels = {'month_calender': '', 'revenue_usd': 'Revenue (USD)'},
               template='simple_white')

fig1.update_yaxes(
    tickprefix='$', ticks='outside'
)

fig1.update_xaxes(
    hoverformat='%b %Y'
)

fig1.update_traces(
    mode='markers+lines', hovertemplate=None
)

fig1.update_layout(
    font_family='rockwell',
    hovermode='x unified',
    title = {
        'text': 'Apple Sales Revenue Over Time <br> <sup style="font-size:0.8em;color:gray;">Hover over data points to see specific USD values</sup>',
        'x': 0.5
    }
)

fig1.add_shape(
    type='line', line_color='salmon', line_width=3, opacity=1, line_dash='dot',
    x0=df_fig1['month_calender'].min(), x1=df_fig1['month_calender'].max(), 
    y0=df_fig1['revenue_usd'].mean(), y1=df_fig1['revenue_usd'].mean(),

)

fig1.add_annotation(
    x=df_fig1['month_calender'].max(), y=df_fig1['revenue_usd'].mean(),
    text='Average Revenue', showarrow=False, yshift=-10, font_color='salmon'
)

fig1.update_traces(mode='markers+lines', hovertemplate=None)


fig1.show()

In [220]:
# Bar chart for sales per year
df_fig2 = df.groupby(df['year'])['revenue_usd'].sum().reset_index()
df_fig2['year'] = df_fig2['year'].astype(str)
df_fig2['revenue_millions'] = df_fig2['revenue_usd'] / 1e6


fig2 = px.bar(
    df_fig2,
    x = 'year',
    y = 'revenue_usd',
    labels = {'year':'Year', 'revenue_usd':'Revenue (USD)'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig2['revenue_millions'].map('${:.1f}M'.format)
)

fig2.update_layout(
    font_family = 'rockwell',
        title={
        "text": "Total Revenue by Year<br><sup style='font-size:12px; color:gray;'>Hover over bars to see specific USD values</sup>",
        "x": 0.5
    }
)

fig2.update_traces(
    hovertemplate='$%{y:,.2f} USD<extra></extra>',
)

fig2.update_yaxes(
    tickprefix='$', ticks='outside'
)


fig2.show()

In [228]:
# Bar chart for units sold per year
df_fig3 = df.groupby('year')['units_sold'].sum().reset_index()
df_fig3['year'] = df_fig3['year'].astype(str)  

fig3 = px.bar(
    df_fig3,
    x = 'year',
    y = 'units_sold',
    labels = {'year':'Year', 'units_sold':'Units Sold'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig3['units_sold'].map('{:.0f}'.format)
)

fig3.update_layout(
    font_family = 'rockwell',
    title = {
        'text': 'Total Units Sold by Year',
        'x': 0.5
    }
)

fig3.update_yaxes(
    ticks='outside',
    tickformat='~s'
)

fig3.update_traces(
    hovertemplate='%{y:.0f} Units<extra></extra>'
)

fig3.show()

In [232]:
# Bar chart for average discount percentage per year
df_fig4 = df.groupby('year')['discount_pct'].mean().reset_index()
df_fig4['year'] = df_fig4['year'].astype(str)

fig4 = px.bar(
    df_fig4,
    x = 'year',
    y = 'discount_pct',
    labels = {'year':'Year', 'discount_pct':'Average Discount (%)'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig4['discount_pct'].map('{:.1f}%'.format)
)

fig4.update_layout(
    font_family = 'rockwell',
    title = {
        'text': 'Average Discount Percentage by Year',
        'x': 0.5
    }
)

fig4.update_yaxes(
    ticks='outside',
    tickformat='.1f%',
    range=[0, df_fig4['discount_pct'].max() * 1.2]
)

fig4.update_traces(
    hovertemplate='%{y:.1f}%<extra></extra>'
)

fig4.show()

# Geographic performance

In [287]:
# Pie chart for revenue per region
df_pie = df.groupby('region')[['revenue_usd', 'units_sold']].sum().reset_index()
pie1 = px.pie(
    df_pie, 
    values='revenue_usd', 
    names='region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)
              
        

pie1.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Revenue by Region <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific USD values</sup>',
        'x': 0.5,
    }
)

pie1.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='$%{value:,.2f} USD<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)



pie1.show()

In [285]:
df_map = df.groupby(['country', 'region'])['revenue_usd'].sum().reset_index()


fig_map = go.Figure(data=go.Choropleth(
    locations=df_map['country'],
    z = df_map['revenue_usd'],
    locationmode='country names',
    text=df_map['country'],
    colorscale='Blues',
    marker_line_color='black',
    marker_line_width=0.5,
    colorbar_ticksuffix='$',
    colorbar_title='Revenue (USD)'
))

fig_map.update_layout(
    title= {
        'text': 'Revenue by Country (2022-2025) <br> <sup style="font-size:12px;color:gray">Hover over countries to see specific USD values</sup>',
        'x': 0.5
    },
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    ),
    annotations=[
        dict(
            x=0.5,
            y=-0.1,
            xref='paper',
            yref='paper',
            text='* Note: Dataset only contains data for 47 countries, so some regions may be underrepresented.',
            showarrow=False,
            font=dict(size=10, color='gray')
        )
    ],
    font_family='rockwell')

fig_map.update_traces(
    hovertemplate='%{text}: $%{z:,.2f} USD<extra></extra>'
)


fig_map.show()

In [292]:
pie2 = px.pie(
    df_pie, 
    values='units_sold', 
    names='region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)

pie2.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Units Sold by Region <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific unit values</sup>',
        'x': 0.5,
    }
)

pie2.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='%{value:.0f} Units<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)

pie2.show()

In [327]:
top10_country_revenue = df.groupby('country')['revenue_usd'].mean().reset_index().sort_values(by='revenue_usd', ascending=False).head(10)

top10_countries = go.Figure(data=[go.Table(
    header=dict(values=['Country', 'Revenue (USD)'],
                fill_color='steelblue',
                font=dict(color='white', 
                          size=12,
                          family='rockwell'),),
    cells=dict(values=[top10_country_revenue['country'], top10_country_revenue['revenue_usd'].map('${:,.2f}'.format)],
                fill_color="#E5E6E6",
                font = dict(color='black',
                            size = 11,
                            family='rockwell')),)      
    ]
)
                          
top10_countries.update_layout(
    title = 'Top 10 Countries by Average Revenue (2022-2025)',
    title_font_family='rockwell',
    title_x=0.5,
)

top10_countries.show()


# Product & Pricing Analysis

In [181]:
# Pie chart for revenue per region
df_fig2 = df.groupby('category')['revenue_usd'].sum().reset_index()
fig2 = px.pie(
    df_fig2, 
    values='revenue_usd', 
    names='category',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)
              
        

fig2.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Revenue by Product Category <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific USD values</sup>',
        'x': 0.5,
    }
)

fig2.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='$%{value:,.2f} USD<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)



fig2.show()

# Customer Insights

# Returns and Satisfaction